In [8]:
import numpy as np
import pandas as pd
import seaborn as sns

import os
from pathlib import Path
import matplotlib.pyplot as plt

import sklearn
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, accuracy_score
from scipy.stats import zscore, pearsonr, uniform
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, StratifiedKFold, RandomizedSearchCV
from sklearn.decomposition import PCA

from scipy.io import loadmat

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import KBinsDiscretizer, StandardScaler
from sklearn.cluster import KMeans

In [9]:
#Load and merge data
BASE_DIR = Path("../../data")
RAW_DIR = BASE_DIR / "raw" / "widsdatathon2025"
FEATURE_ENGINEERED_DIR = BASE_DIR / "feature engineered"

# Ensure directory exists
FEATURE_ENGINEERED_DIR.mkdir(parents=True, exist_ok=True)

df_quant = pd.read_excel(RAW_DIR / "TRAIN/TRAIN_QUANTITATIVE_METADATA.xlsx")
df_cat = pd.read_excel(RAW_DIR / "TRAIN/TRAIN_CATEGORICAL_METADATA.xlsx")
df_labels = pd.read_excel(RAW_DIR / "TRAIN/TRAINING_SOLUTIONS.xlsx")
df_fcm= pd.read_csv(RAW_DIR / "TRAIN/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES.csv")

In [10]:
df = df_cat.merge(df_labels, on="participant_id", how="left")
df_quant = df_quant.merge(df_labels, on="participant_id", how="left")
df_fcm = df_fcm.merge(df_labels, on="participant_id", how="left")

In [11]:
def classify_occupation(occupation):
    return 0 if occupation >= 40 else 1
#look at parent occupation, classify as professional or not

df['parent_occupation_1'] = df['Barratt_Barratt_P1_Occ'].apply(classify_occupation)
df['parent_occupation_2'] = df['Barratt_Barratt_P2_Occ'].apply(classify_occupation)

# Feature 2: Parent Occupation Similarity
df['parent_occupation_similarity'] = (df['parent_occupation_1'] == df['parent_occupation_2']).astype(int)

In [12]:
# Feature 3: Minority Status
df['minority_status'] = df['PreInt_Demos_Fam_Child_Race'].apply(lambda x: 1 if x in [1, 2, 4, 5, 6, 7, 8, 9] else 0)

In [13]:
# Feature 4: Binning Color Vision Scores
color_bins = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
df_quant['color_vision_bin'] = color_bins.fit_transform(df_quant[['ColorVision_CV_Score']])

c:\Users\linds\OneDrive\Desktop\WIDS\venv\lib\site-packages\sklearn\preprocessing\_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(


In [14]:
# Feature 5: ADHD Risk Scores (Deviation from Mean)
scaler = StandardScaler()
df_quant['adhd_risk_deviation'] = scaler.fit_transform(df_quant[['ADHD_Outcome']])

In [15]:
# Feature 6: Clustering Features using K-Means
feature_cols = ['EHQ_EHQ_Total', 'SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Conduct_Problems', 'SDQ_SDQ_Difficulties_Total']
kmeans = KMeans(n_clusters=3, random_state=42)
df_quant['feature_cluster'] = kmeans.fit_predict(df_quant[feature_cols])

In [16]:
# Feature 7: Interaction Term (Parent Education x Occupation)
df['parent_edu_occ'] = df['Barratt_Barratt_P1_Edu'] * (df['parent_occupation_1'] + df['parent_occupation_2'])

In [17]:
# Save Engineered Data
df_quant.to_csv(FEATURE_ENGINEERED_DIR / "wids_feature_engineered_quantitative.csv", index=False)
df_cat.to_csv(FEATURE_ENGINEERED_DIR / "wids_feature_engineered_categorical.csv", index=False)
df_fcm.to_csv(FEATURE_ENGINEERED_DIR / "wids_feature_engineered_fcm.csv", index=False)

print("Feature engineering complete. Data saved.")

Feature engineering complete. Data saved.
